# Region-based quantification of 1P widefield movies

Take a trial-averaged movie `(N, 1, H, W)`, pick brain regions (hand-drawn or
loaded from json), mask the movie to each region, then optionally reduce to a
per-region statistic.

- masked movies: `dict{name: (N, H, W)}`, pixels outside the region are `NaN`
- `quantify(..., axis="spatial")` -> `(N,)` time-series per region
- `quantify(..., axis="frames")`  -> `(H, W)` map per region

Regions must be **mutually exclusive**; overlaps raise `ValueError`.

In [ ]:
from piepy.sensory.visual.visualSession import VisualSession
from piepy.imaging.onep.widefield.onepAnalysis import OnePAnalysis
from piepy.imaging.onep.widefield.regions import (
    reference_frame,
    draw_regions,
    save_regions,
    load_regions,
    apply_regions,
    quantify,
)

In [ ]:
vis_sesh = VisualSession('240207_KC149__1P_KC', load_flag=False)
run = vis_sesh.runs[0]
o = OnePAnalysis(run.data.data, run.paths.onepcam)
trial_avg = o.trial_avg(batch_count=3)  # (N, 1, H, W)

## 1. Pick regions

### Option A - hand-draw on a reference frame
Needs an interactive backend. Run `%matplotlib qt` (or `%matplotlib widget`)
first. Default backdrop is the mean projection; pass `source=` a tif path or
array to use an anatomical image instead.

In [ ]:
%matplotlib qt
ref = reference_frame(trial_avg)                 # or reference_frame(trial_avg, source='anat.tif')
regions = draw_regions(ref)                       # draw, <enter>, name; blank name to stop
save_regions(regions, 'rois.json')

### Option B - load vertices from json
json format: `{"region_name": [[x, y], ...], ...}`

In [ ]:
regions = load_regions('rois.json')

## 2. Mask movie to each region -> K masked movies `(N, H, W)`

In [ ]:
masked = apply_regions(trial_avg, regions)       # dict{name: (N, H, W)}, NaN outside region
{k: v.shape for k, v in masked.items()}

## 3. Quantify
### 3a. Over space -> time-series `(N,)` per region

In [ ]:
ts_mean = quantify(masked, 'mean', axis='spatial')
ts_max = quantify(masked, 'max', axis='spatial')

### 3b. Over frames -> map `(H, W)` per region (each pixel = stat of N frames)

In [ ]:
map_mean = quantify(masked, 'mean', axis='frames')

### 3c. Custom function with keyword passthrough
`stat` may be any callable `func(arr, axis=<int|tuple>, **kwargs)`; extra kwargs
are forwarded.

In [ ]:
import numpy as np

# built-in numpy reducer with an argument
p90 = quantify(masked, np.nanpercentile, axis='frames', q=90)

# your own reducer
def range_stat(arr, axis, clip=None):
    r = np.nanmax(arr, axis=axis) - np.nanmin(arr, axis=axis)
    return np.clip(r, 0, clip) if clip is not None else r

rng = quantify(masked, range_stat, axis='spatial', clip=5000)

## 4. Quick look

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
for name, y in ts_mean.items():
    ax.plot(y, label=name)
ax.set_xlabel('frame')
ax.set_ylabel('mean activity')
ax.legend()